# The Heartbeat at the Edge of Chaos
## What 21,000 ECGs Reveal About Cardiac Complexity

---

**TL;DR:** We apply IRASA spectral decomposition to 21,797 twelve-lead ECGs from PTB-XL and extract a single physics-based parameter — the spectral exponent β — per lead. Four results:

1. **Diagnostic landscape:** Each cardiac condition has a unique β-fingerprint. Conduction Disturbances are the strongest outlier (Cohen's d ≈ 1.0).

2. **Spectral anatomy:** The 12-lead β-vector encodes the anatomy of the conduction system. A classifier distinguishes CLBBB from CRBBB with **AUC = 0.984** from 54 interpretable features — approaching deep learning performance (AUC ~0.99 on raw waveform) without any waveform morphology. Radar plots of β per lead literally draw where the lesion is.

3. **Aging bifurcation:** β *falls* with healthy aging (ρ = −0.18) but *rises* in disease. The healthy heart sits near a critical operating point; aging and pathology push it in opposite directions. Beat-to-beat morphological variability independently confirms this complexity loss (ρ = 0.165).

4. **Subclinical detection:** Among ECGs labeled "normal" by cardiologists, β distinguishes pure normals from those with subclinical conduction findings (AUC = 0.76, Cohen's d = −0.55). β catches what the expert decides not to diagnose.

---

### Why this matters

Most ECG analyses focus on *periodic* features — heart rate, QRS duration, ST morphology. But the ECG also contains an *aperiodic* (1/fᵝ) component reflecting the broadband complexity of cardiac electrical activity. This spectral exponent β captures how organized vs. chaotic the heart's electrical dynamics are.

The headline finding: **one number per lead is enough to reconstruct the anatomy of the conduction system, track cardiac aging, and detect subclinical disease.** β doesn't replace deep learning — it explains what deep learning is implicitly learning about cardiac electrophysiology.

## Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from scipy import signal, stats
from scipy.optimize import minimize_scalar
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, confusion_matrix
from sklearn.model_selection import cross_val_predict, KFold
import ast
import os
import warnings
warnings.filterwarnings('ignore')

# Plotting style
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
})

# Color scheme
COLORS = {
    'NORM': '#4CAF50',
    'MI':   '#2196F3',
    'STTC': '#FF9800',
    'CD':   '#E53935',
    'HYP':  '#9C27B0',
}
CD_SUBTYPE_COLORS = {
    'CLBBB': '#E53935', 'CRBBB': '#1565C0', 'IRBBB': '#00BCD4',
    'LAFB':  '#FF9800', '1AVB':  '#9C27B0', 'IVCD':  '#607D8B',
}
LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

print('Libraries loaded.')

In [ ]:
# ============================================================
# Load PTB-XL metadata
# ============================================================
# --- PATH CONFIGURATION ---
# On Kaggle, uncomment the appropriate lines:
#   DATA_DIR = '/kaggle/input/ptb-xl-a-large-publicly-available-electrocardiography-dataset/'
#   PRECOMPUTED_DIR = '/kaggle/input/heartbeat-chaos-precomputed/'
# On Kaggle:
DATA_DIR = '/kaggle/input/ptb-xl-a-large-publicly-available-electrocardiography-dataset/'
PRECOMPUTED_DIR = '/kaggle/input/heartbeat-chaos-precomputed/'

meta = pd.read_csv(os.path.join(DATA_DIR, 'ptbxl_database.csv'), index_col='ecg_id')
scp = pd.read_csv(os.path.join(DATA_DIR, 'scp_statements.csv'), index_col=0)

# Parse SCP codes
def parse_scp(scp_str):
    try:
        return ast.literal_eval(scp_str)
    except:
        return {}

meta['scp_dict'] = meta.scp_codes.apply(parse_scp)

# Map to diagnostic superclass
def get_superclass(scp_dict):
    classes = set()
    for code, likelihood in scp_dict.items():
        if code in scp.index:
            dc = scp.loc[code, 'diagnostic_class']
            if pd.notna(dc):
                classes.add(dc)
    return list(classes)

meta['superclasses'] = meta.scp_dict.apply(get_superclass)
meta['n_super'] = meta.superclasses.apply(len)
meta['clean_superclass'] = meta.superclasses.apply(lambda x: x[0] if len(x) == 1 else None)

print(f'PTB-XL: {len(meta):,} records, {meta.patient_id.nunique():,} patients')
print(f'Age range: {meta.age.min():.0f}-{meta.age.max():.0f}, Sex: {(meta.sex==0).sum():,}F / {(meta.sex==1).sum():,}M')
print(f'\nDiagnostic superclass distribution (single-label):')
print(meta.clean_superclass.value_counts().to_string())

## Step 1: Compute Spectral Exponent β via IRASA

**IRASA** (Wen & Liu, 2016) separates the power spectral density (PSD) into:
- **Aperiodic component** (1/fᵝ background) — broadband complexity
- **Periodic component** (oscillatory peaks) — heart rate harmonics

The key idea: resampling a signal at irrational rates shifts periodic peaks but preserves the aperiodic slope. By averaging across multiple resampling factors, the oscillatory peaks cancel out, leaving a clean estimate of the 1/fᵝ background.

We fit a linear model in log-log space (2–45 Hz) to extract the **spectral exponent β** for each of the 12 ECG leads.

> ⚠️ **Note:** This cell processes all 21,797 records and takes ~40–60 minutes on a standard Kaggle kernel. We provide a pre-computed option below. To run from scratch, set `COMPUTE_FROM_SCRATCH = True`.

In [ ]:
COMPUTE_FROM_SCRATCH = False  # Set True to recompute (takes ~45 min)

FS = 500  # Hz
F_MIN, F_MAX = 2.0, 45.0  # Fitting range
IRASA_HSET = np.arange(1.1, 1.95, 0.05)

def preprocess_lead(sig, fs=500):
    """Bandpass 0.5-100 Hz + 50 Hz notch."""
    sos = signal.butter(4, [0.5, 100], btype='bandpass', fs=fs, output='sos')
    sig = signal.sosfiltfilt(sos, sig)
    b, a = signal.iirnotch(50, Q=30, fs=fs)
    sig = signal.filtfilt(b, a, sig)
    return sig


def compute_irasa_beta(sig, fs=500, f_min=2.0, f_max=45.0):
    """
    Compute spectral exponent via IRASA.
    Returns (beta, r_squared) or (nan, nan) on failure.
    """
    from neurodsp.aperiodic import compute_irasa
    try:
        freqs, pa, pp = compute_irasa(sig, fs=fs, f_range=(0.5, 50), hset=IRASA_HSET)
        mask = (freqs >= f_min) & (freqs <= f_max) & (pa > 0)
        if mask.sum() < 5:
            return np.nan, np.nan
        log_f = np.log10(freqs[mask])
        log_p = np.log10(pa[mask])
        slope, intercept, r_value, _, _ = stats.linregress(log_f, log_p)
        return -slope, r_value**2
    except:
        return np.nan, np.nan


if COMPUTE_FROM_SCRATCH:
    import wfdb
    
    results = []
    total = len(meta)
    
    for i, (ecg_id, row) in enumerate(meta.iterrows()):
        if (i + 1) % 2000 == 0:
            print(f'  Processing {i+1}/{total}...')
        
        try:
            rec = wfdb.rdrecord(os.path.join(DATA_DIR, row.filename_hr))
            ecg = rec.p_signal  # (5000, 12)
        except:
            continue
        
        entry = {'ecg_id': ecg_id}
        betas = []
        
        for j, lead in enumerate(LEAD_NAMES):
            sig = preprocess_lead(ecg[:, j], fs=FS)
            beta, r2 = compute_irasa_beta(sig, fs=FS)
            entry[f'beta_ir_{lead}'] = beta
            entry[f'r2_ir_{lead}'] = r2
            if not np.isnan(beta):
                betas.append(beta)
        
        if betas:
            entry['beta_mean'] = np.mean(betas)
            entry['beta_std'] = np.std(betas)
            entry['beta_median'] = np.median(betas)
            entry['n_valid'] = len(betas)
        
        results.append(entry)
    
    beta_df = pd.DataFrame(results)
    beta_df.to_csv('beta_features.csv', index=False)
    print(f'Computed \u03b2 for {len(beta_df):,} records')

else:
    # Load pre-computed features
    beta_df = pd.read_csv(os.path.join(PRECOMPUTED_DIR, 'beta_features.csv'))
    print(f'Loaded pre-computed \u03b2 for {len(beta_df):,} records')
    print(f'Columns: {len(beta_df.columns)}, Leads: {sum(1 for c in beta_df.columns if c.startswith("beta_ir_"))}')

In [ ]:
# ============================================================
# Merge beta features with metadata
# ============================================================
df = meta.join(beta_df.set_index('ecg_id'), how='inner')
df = df[df.beta_mean.notna()].copy()

# Compute regional averages
regions = {
    'anterior': ['V1', 'V2', 'V3', 'V4'],
    'lateral':  ['I', 'aVL', 'V5', 'V6'],
    'inferior': ['II', 'III', 'aVF'],
}
for region, leads in regions.items():
    cols = [f'beta_ir_{l}' for l in leads]
    df[f'beta_{region}'] = df[cols].mean(axis=1)

df['beta_regional_div'] = (df[['beta_anterior','beta_lateral','beta_inferior']].max(axis=1) -
                            df[['beta_anterior','beta_lateral','beta_inferior']].min(axis=1))

# Additional derived features
lead_betas = [f'beta_ir_{l}' for l in LEAD_NAMES]
r2_cols = [f'r2_ir_{l}' for l in LEAD_NAMES]
sp_cols = [f'beta_sp_{l}' for l in LEAD_NAMES]

df['beta_iqr'] = df[lead_betas].quantile(0.75, axis=1) - df[lead_betas].quantile(0.25, axis=1)
df['beta_cv'] = df['beta_std'] / df['beta_mean'].abs()
df['beta_skew'] = df[lead_betas].skew(axis=1)
df['beta_range'] = df[lead_betas].max(axis=1) - df[lead_betas].min(axis=1)
df['r2_std'] = df[r2_cols].std(axis=1)
df['sex_num'] = df['sex'].astype(float)
df['age_x_beta'] = df['age'] * df['beta_mean']

# NORM-only cohort
norm = df[df.clean_superclass == 'NORM'].copy()
norm = norm[(norm.age >= 18) & (norm.age <= 95)].copy()

# Extract CD subtypes (for Section 4)
cd_subtypes_of_interest = ['CLBBB', 'CRBBB', 'LAFB', '1AVB', 'IVCD', 'IRBBB']

cd_subtype_list = []
for ecg_id, row in df.iterrows():
    codes = row['scp_dict']
    for subtype in cd_subtypes_of_interest:
        if subtype in codes and codes[subtype] >= 80:
            cd_subtype_list.append({'ecg_id': ecg_id, 'cd_subtype': subtype})
            break

cd_sub_df = pd.DataFrame(cd_subtype_list).drop_duplicates(subset='ecg_id').set_index('ecg_id')
df = df.join(cd_sub_df, how='left')

print(f'Total merged: {len(df):,} records')
print(f'NORM cohort (18-95y): {len(norm):,} records')
print(f'\u03b2 summary (NORM): median = {norm.beta_mean.median():.3f}, '
      f'mean = {norm.beta_mean.mean():.3f}, std = {norm.beta_mean.std():.3f}')
print(f'\nCD subtypes extracted: {cd_sub_df.cd_subtype.value_counts().to_string()}')

---
## Result 1: The β-Landscape of Cardiac Diagnoses

Each of the 5 diagnostic superclasses in PTB-XL has a characteristic spectral exponent. The healthy heart (β ≈ 1.76) defines a reference point. Conduction Disturbances (CD) diverge most strongly.

In [ ]:
# ============================================================
# Figure 1: Diagnostic Beta Landscape
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel A: Violin plot by superclass ---
ax = axes[0]
order = ['NORM', 'STTC', 'MI', 'HYP', 'CD']
clean = df[df.clean_superclass.isin(order)].copy()

positions = range(len(order))
for i, cls in enumerate(order):
    vals = clean[clean.clean_superclass == cls].beta_mean.dropna()
    parts = ax.violinplot(vals, positions=[i], showmeans=False, showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(COLORS[cls])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')

ax.axhline(norm.beta_mean.median(), color='gray', ls='--', alpha=0.5,
           label=f'NORM median = {norm.beta_mean.median():.2f}')
ax.set_xticks(positions)
ax.set_xticklabels(order, fontsize=11)
ax.set_ylabel('Spectral exponent \u03b2')
ax.set_title('A. \u03b2 by diagnostic superclass', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add sample sizes
for i, cls in enumerate(order):
    n = (clean.clean_superclass == cls).sum()
    ax.text(i, ax.get_ylim()[0] + 0.05, f'n={n:,}', ha='center', fontsize=8, color='gray')


# --- Panel B: Cohen's d effect sizes ---
ax = axes[1]
norm_vals = clean[clean.clean_superclass == 'NORM'].beta_mean.dropna()
n1 = len(norm_vals)
s1 = norm_vals.std()

effect_sizes = []
for cls in ['MI', 'STTC', 'HYP', 'CD']:
    cls_vals = clean[clean.clean_superclass == cls].beta_mean.dropna()
    n2 = len(cls_vals)
    s2 = cls_vals.std()
    # Weighted pooled std (same as pingouin)
    pooled_std = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
    d = (cls_vals.mean() - norm_vals.mean()) / pooled_std
    _, p = stats.mannwhitneyu(norm_vals, cls_vals)
    effect_sizes.append({'class': cls, 'd': d, 'p': p})

es_df = pd.DataFrame(effect_sizes).sort_values('d')
bars = ax.barh(range(len(es_df)), es_df.d.values, color=[COLORS[c] for c in es_df['class']], alpha=0.8)
ax.set_yticks(range(len(es_df)))
ax.set_yticklabels([f"{r['class']}  (d={r['d']:.2f})" for _, r in es_df.iterrows()], fontsize=10)
ax.axvline(0, color='black', lw=0.5)
ax.axvline(0.5, color='gray', ls=':', alpha=0.5, label='medium (0.5)')
ax.axvline(0.8, color='gray', ls='--', alpha=0.5, label='large (0.8)')
ax.set_xlabel("Cohen's d (vs NORM)")
ax.set_title("B. Effect size: disease vs healthy", fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='x', alpha=0.3)


# --- Panel C: Spatial fingerprint (CLBBB) ---
ax = axes[2]

# Get CLBBB, CRBBB, and NORM spatial profiles
clbbb = df[df.scp_dict.apply(lambda x: 'CLBBB' in x and x.get('CLBBB', 0) >= 80)]
crbbb = df[df.scp_dict.apply(lambda x: 'CRBBB' in x and x.get('CRBBB', 0) >= 80)]

for label, subset, color, marker in [
    ('NORM', norm, COLORS['NORM'], 'o'),
    ('CLBBB', clbbb, '#E53935', 's'),
    ('CRBBB', crbbb, '#1565C0', '^'),
]:
    medians = [subset[f'beta_ir_{l}'].median() for l in LEAD_NAMES]
    q25 = [subset[f'beta_ir_{l}'].quantile(0.25) for l in LEAD_NAMES]
    q75 = [subset[f'beta_ir_{l}'].quantile(0.75) for l in LEAD_NAMES]
    x = range(len(LEAD_NAMES))
    ax.plot(x, medians, f'{marker}-', color=color, linewidth=2, markersize=6,
            label=f'{label} (n={len(subset):,})')
    ax.fill_between(x, q25, q75, alpha=0.15, color=color)

# Highlight anterior leads
ax.axvspan(6, 9, alpha=0.08, color='red', label='Anterior leads (V1-V4)')
ax.set_xticks(range(len(LEAD_NAMES)))
ax.set_xticklabels(LEAD_NAMES, fontsize=9)
ax.set_ylabel('\u03b2 (median)')
ax.set_title('C. Spatial fingerprint: conduction blocks', fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)

plt.suptitle('Figure 1. The \u03b2-landscape of cardiac diagnoses',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Key finding #1

- **NORM** β ≈ 1.76 — the healthy cardiac operating point
- **CD** (Conduction Disturbance) is the strongest outlier: **d ≈ 1.0**, the only superclass reaching "large" effect size territory
- **Spatial fingerprint matches anatomy:** CLBBB shows β ≈ 3.5 specifically in V1–V4 (anterior leads overlying the septum and left ventricle), while CRBBB shows elevation primarily in V1 — exactly where the anatomical lesion is
- This is **not just a statistical difference** — it's a physically meaningful mapping from electrical lesion to spectral signature

---
## Result 2: Spectral Anatomy of the Conduction System

**This is the headline finding.** The 12-lead β-vector doesn't just *differ* between pathologies — it encodes *where* in the conduction system the lesion is. Each type of bundle branch block, fascicular block, and AV block produces a unique spatial β-fingerprint that maps directly onto cardiac anatomy.

We classify 6 CD subtypes (CLBBB, CRBBB, IRBBB, LAFB, 1AVB, IVCD) using only β-features, and test the pairwise distinction CLBBB vs CRBBB — left vs. right bundle branch block.

In [ ]:
# ============================================================
# Figure 2: Spectral Anatomy — CD Subtype Classification
# ============================================================

# --- Prepare CD subtype data ---
cd_data = df[df.cd_subtype.notna()].copy()

# Feature set for classification
feat_cols = ['age', 'sex_num', 'beta_mean', 'beta_std', 'beta_median', 'delta',
             'beta_anterior', 'beta_lateral', 'beta_inferior', 'beta_regional_div',
             'r2_mean', 'r2_std', 'beta_iqr', 'beta_cv', 'beta_skew', 'beta_range',
             'age_x_beta', 'beta_sp_mean'] + lead_betas + r2_cols + sp_cols
avail = [f for f in feat_cols if f in cd_data.columns]

# --- CLBBB vs CRBBB binary classification ---
bbb_data = cd_data[cd_data.cd_subtype.isin(['CLBBB', 'CRBBB'])].copy()
bbb_data['label'] = (bbb_data.cd_subtype == 'CLBBB').astype(int)

bbb_train = bbb_data[bbb_data.strat_fold.isin(range(1, 9))]
bbb_test = bbb_data[bbb_data.strat_fold.isin([9, 10])]

tr = bbb_train[avail + ['label']].dropna()
te = bbb_test[avail + ['label']].dropna()
X_tr, y_tr = tr[avail].values, tr['label'].values
X_te, y_te = te[avail].values, te['label'].values

scaler_bbb = StandardScaler()
X_tr_s = scaler_bbb.fit_transform(X_tr)
X_te_s = scaler_bbb.transform(X_te)

gbm_bbb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                      subsample=0.8, random_state=42)
gbm_bbb.fit(X_tr_s, y_tr)
y_prob_bbb = gbm_bbb.predict_proba(X_te_s)[:, 1]
bbb_auc = roc_auc_score(y_te, y_prob_bbb)
bbb_fpr, bbb_tpr, _ = roc_curve(y_te, y_prob_bbb)

print(f'CLBBB vs CRBBB: AUC = {bbb_auc:.3f}')
print(f'  Train: {len(tr)} (CLBBB={y_tr.sum()}, CRBBB={len(y_tr)-y_tr.sum()})')
print(f'  Test:  {len(te)} (CLBBB={y_te.sum()}, CRBBB={len(y_te)-y_te.sum()})')

# --- 6-class CD subtype classification ---
cd_train = cd_data[cd_data.strat_fold.isin(range(1, 9))]
cd_test = cd_data[cd_data.strat_fold.isin([9, 10])]

tr6 = cd_train[avail + ['cd_subtype']].dropna()
te6 = cd_test[avail + ['cd_subtype']].dropna()
X_tr6, y_tr6 = tr6[avail].values, tr6['cd_subtype'].values
X_te6, y_te6 = te6[avail].values, te6['cd_subtype'].values

scaler6 = StandardScaler()
X_tr6_s = scaler6.fit_transform(X_tr6)
X_te6_s = scaler6.transform(X_te6)

gbm6 = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                   subsample=0.8, random_state=42)
gbm6.fit(X_tr6_s, y_tr6)
y_pred6 = gbm6.predict(X_te6_s)
y_prob6 = gbm6.predict_proba(X_te6_s)

from sklearn.preprocessing import label_binarize
classes6 = gbm6.classes_
y_te6_bin = label_binarize(y_te6, classes=classes6)
class_order = list(classes6)

per_class_auc6 = {}
for i, cls in enumerate(classes6):
    try:
        per_class_auc6[cls] = roc_auc_score(y_te6_bin[:, i], y_prob6[:, i])
    except:
        per_class_auc6[cls] = np.nan

macro_auc6 = np.nanmean(list(per_class_auc6.values()))
print(f'\n6-class CD subtype: Macro AUC = {macro_auc6:.3f}')
for cls in cd_subtypes_of_interest:
    if cls in per_class_auc6:
        print(f'  {cls:8s}: AUC = {per_class_auc6[cls]:.3f}')

# ============================================================
# FIGURE 2: 4 panels
# ============================================================
fig = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3,
                        width_ratios=[1.3, 1, 1])

# --- Panel A: Spatial fingerprint (line plot) ---
ax = fig.add_subplot(gs[0, :2])

# NORM reference
norm_medians = [norm[f'beta_ir_{l}'].median() for l in LEAD_NAMES]
ax.plot(range(len(LEAD_NAMES)), norm_medians, 'o--', color=COLORS['NORM'],
        linewidth=1.5, markersize=4, alpha=0.5, label=f'NORM (n={len(norm):,})')

for subtype in ['CLBBB', 'CRBBB', 'IRBBB', 'LAFB', '1AVB', 'IVCD']:
    sub = cd_data[cd_data.cd_subtype == subtype]
    if len(sub) < 10:
        continue
    medians = [sub[f'beta_ir_{l}'].median() for l in LEAD_NAMES]
    q25 = [sub[f'beta_ir_{l}'].quantile(0.25) for l in LEAD_NAMES]
    q75 = [sub[f'beta_ir_{l}'].quantile(0.75) for l in LEAD_NAMES]
    ax.plot(range(len(LEAD_NAMES)), medians, 'o-',
            color=CD_SUBTYPE_COLORS[subtype], linewidth=2, markersize=5,
            label=f'{subtype} (n={len(sub):,})')
    ax.fill_between(range(len(LEAD_NAMES)), q25, q75, alpha=0.08,
                    color=CD_SUBTYPE_COLORS[subtype])

ax.axvspan(6, 9, alpha=0.06, color='red')
ax.text(7.5, ax.get_ylim()[0] + 0.15 if ax.get_ylim()[0] < 1.5 else 1.5,
        'V1-V4 (anterior)', ha='center', fontsize=8, color='red', alpha=0.7)
ax.set_xticks(range(len(LEAD_NAMES)))
ax.set_xticklabels(LEAD_NAMES, fontsize=10)
ax.set_ylabel('\u03b2 (median)', fontsize=11)
ax.set_title('A. Spatial \u03b2-fingerprint by conduction disturbance subtype\n'
             'Each lesion type has a unique anatomical pattern', fontweight='bold')
ax.legend(fontsize=8, loc='upper left', ncol=2)
ax.grid(alpha=0.3)

# --- Panel B: Per-subtype AUC bar chart ---
ax = fig.add_subplot(gs[0, 2])

sorted_subtypes = sorted(per_class_auc6.keys(),
                         key=lambda k: per_class_auc6.get(k, 0), reverse=True)
aucs_sorted = [per_class_auc6[c] for c in sorted_subtypes]
bar_colors = [CD_SUBTYPE_COLORS.get(c, 'gray') for c in sorted_subtypes]

bars = ax.bar(range(len(sorted_subtypes)), aucs_sorted, color=bar_colors,
              alpha=0.85, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(sorted_subtypes)))
ax.set_xticklabels(sorted_subtypes, fontsize=9)
ax.set_ylabel('AUC (one-vs-rest)', fontsize=11)
ax.set_title(f'B. Per-subtype AUC (macro = {macro_auc6:.2f})\n'
             f'CLBBB nearly perfect', fontweight='bold')
ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
ax.set_ylim(0.4, 1.05)
ax.grid(axis='y', alpha=0.3)
for bar, auc_val in zip(bars, aucs_sorted):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{auc_val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- Panel C: CLBBB vs CRBBB ROC ---
ax = fig.add_subplot(gs[1, 0])
ax.plot(bbb_fpr, bbb_tpr, linewidth=2.5, color='#C62828',
        label=f'GBM (AUC = {bbb_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'C. CLBBB vs CRBBB: AUC = {bbb_auc:.3f}\n'
             f'Left vs right bundle branch block', fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)

# --- Panel D: Feature importance for CLBBB vs CRBBB ---
ax = fig.add_subplot(gs[1, 1])
importances = gbm_bbb.feature_importances_
feat_imp = pd.DataFrame({'feature': avail, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=True).tail(12)

colors_imp = []
for f in feat_imp.feature:
    if f.startswith('beta_ir_V'):
        colors_imp.append('#E53935')
    elif f.startswith('beta_ir_'):
        colors_imp.append('#FF9800')
    elif f.startswith('r2_ir_'):
        colors_imp.append('#2196F3')
    else:
        colors_imp.append('#9E9E9E')

ax.barh(range(len(feat_imp)), feat_imp.importance.values, color=colors_imp, alpha=0.85)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(feat_imp.feature.values, fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('D. What separates left from right?\n'
             'Precordial leads dominate', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# --- Panel E: Radar plots (CLBBB vs CRBBB vs NORM) ---
ax = fig.add_subplot(gs[1, 2], projection='polar')

angles = np.linspace(0, 2*np.pi, len(LEAD_NAMES), endpoint=False).tolist()
angles += angles[:1]

for label, subset, color in [
    ('NORM', norm, COLORS['NORM']),
    ('CLBBB', cd_data[cd_data.cd_subtype=='CLBBB'], '#E53935'),
    ('CRBBB', cd_data[cd_data.cd_subtype=='CRBBB'], '#1565C0'),
]:
    meds = [subset[f'beta_ir_{l}'].median() for l in LEAD_NAMES] + \
           [subset[f'beta_ir_{LEAD_NAMES[0]}'].median()]
    ax.plot(angles, meds, 'o-', color=color, linewidth=2, markersize=4,
            label=f'{label} (n={len(subset):,})')
    ax.fill(angles, meds, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LEAD_NAMES, fontsize=7)
ax.set_title('E. Radar: \u03b2 maps the lesion\n', fontweight='bold', fontsize=10, pad=20)
ax.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.suptitle('Figure 2. Spectral anatomy of the conduction system',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Key finding #2 — Headline result

- **CLBBB vs CRBBB: AUC = 0.984** — from 54 interpretable β-features, without any waveform morphology
- For context: deep learning on raw 12-lead ECG achieves AUC ~0.99 for this task using ~60,000 inputs. We reach 98.4% of that performance with **one number per lead**
- **6-class CD subtype macro AUC = 0.86** — CLBBB (0.97), LAFB (0.89), IRBBB (0.89), CRBBB (0.88)
- **The radar plots tell the story visually:**
  - CLBBB → massive β spike in V1-V4 (the left bundle passes through the interventricular septum, directly beneath V1-V4)
  - CRBBB → moderate β elevation in V1 only (right bundle is close to V1)
  - LAFB → elevation centered on aVL (anterior fascicle projects to the lateral wall)
- **The model literally reconstructs the anatomy of the conduction system** from spectral exponents. This is not pattern matching — it's physics-informed imaging.
- Feature importance confirms: precordial leads (V1-V6) dominate the CLBBB-vs-CRBBB distinction, exactly as predicted by cardiac anatomy

---
## Result 3: The Aging Bifurcation

Aging and disease affect β in **opposite directions**. The healthy heart gradually loses complexity with age (β ↓), while pathology increases β. This creates a bifurcation where the healthy operating point is a saddle between two failure modes.

Beat-to-beat morphological variability — an independent metric at a completely different timescale — confirms the same aging story.

In [ ]:
# ============================================================
# Figure 3: Aging & Bifurcation
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel A: Beta vs Age (NORM only) ---
ax = axes[0]

bins = np.arange(20, 91, 5)
norm['age_bin'] = pd.cut(norm.age, bins=bins)
binned = norm.groupby('age_bin').agg(
    age_mid=('age', 'mean'),
    beta_mean_avg=('beta_mean', 'mean'),
    beta_mean_se=('beta_mean', lambda x: x.std() / np.sqrt(len(x))),
    n=('beta_mean', 'count'),
).dropna()

rng = np.random.RandomState(42)
idx = rng.choice(len(norm), min(3000, len(norm)), replace=False)
ax.scatter(norm.age.values[idx], norm.beta_mean.values[idx],
           alpha=0.08, s=8, c=COLORS['NORM'], rasterized=True)
ax.errorbar(binned.age_mid, binned.beta_mean_avg, yerr=binned.beta_mean_se * 1.96,
            fmt='ko-', markersize=5, linewidth=1.5, capsize=3, zorder=5, label='Binned mean \u00b1 95% CI')

rho_age, p_age = stats.spearmanr(norm.age, norm.beta_mean)
ax.set_xlabel('Age (years)')
ax.set_ylabel('\u03b2 (mean across leads)')
ax.set_title(f'A. \u03b2 declines with healthy aging\n\u03c1 = {rho_age:.3f}, p < 10\u207b\u2076\u2070', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Panel B: Breakpoint analysis ---
ax = axes[1]

age_vals = norm.age.values.astype(float)
bstd_vals = norm.beta_std.values.astype(float)
valid = np.isfinite(age_vals) & np.isfinite(bstd_vals)
age_v, bstd_v = age_vals[valid], bstd_vals[valid]

def segmented_rss(bp, y_vals):
    left = age_v <= bp
    right = age_v > bp
    if left.sum() < 50 or right.sum() < 50:
        return 1e10
    rss = 0
    for mask in [left, right]:
        s, i, _, _, _ = stats.linregress(age_v[mask], y_vals[mask])
        rss += np.sum((y_vals[mask] - (i + s * age_v[mask]))**2)
    return rss

bp_candidates = np.arange(30, 65, 0.5)
rss_values = [segmented_rss(b, bstd_v) for b in bp_candidates]
bp = bp_candidates[np.argmin(rss_values)]

left_mask = age_v <= bp
right_mask = age_v > bp
sl, il, _, _, _ = stats.linregress(age_v[left_mask], bstd_v[left_mask])
sr, ir, _, _, _ = stats.linregress(age_v[right_mask], bstd_v[right_mask])

binned_bp = norm.groupby('age_bin').agg(
    age_mid=('age', 'mean'), bstd_avg=('beta_std', 'mean'),
    bstd_se=('beta_std', lambda x: x.std() / np.sqrt(len(x))),
).dropna()

ax.errorbar(binned_bp.age_mid, binned_bp.bstd_avg, yerr=binned_bp.bstd_se * 1.96,
            fmt='ko-', markersize=5, linewidth=1.5, capsize=3, zorder=5)
x_left = np.linspace(18, bp, 50)
x_right = np.linspace(bp, 90, 50)
ax.plot(x_left, il + sl * x_left, 'b-', linewidth=2, label=f'<{bp:.0f}y')
ax.plot(x_right, ir + sr * x_right, 'r-', linewidth=2, label=f'>{bp:.0f}y')
ax.axvline(bp, color='gray', ls=':', alpha=0.7, label=f'Breakpoint: {bp:.0f}y')
ax.set_xlabel('Age (years)')
ax.set_ylabel('\u03b2 spatial heterogeneity (std)')
ax.set_title(f'B. Heterogeneity accelerates at ~{bp:.0f}y', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Panel C: Bifurcation diagram ---
ax = axes[2]

age_decades = [(20,35,'Young\n(20-35)'), (36,50,'Middle\n(36-50)'),
               (51,65,'Senior\n(51-65)'), (66,90,'Elderly\n(66-90)')]
x_aging, y_aging, labels_aging = [], [], []
for lo, hi, label in age_decades:
    sub = norm[(norm.age >= lo) & (norm.age <= hi)]
    x_aging.append(sub.age.mean()); y_aging.append(sub.beta_mean.median()); labels_aging.append(label)

ax.plot(x_aging, y_aging, 'o-', color=COLORS['NORM'], linewidth=2.5, markersize=10,
        label='Aging (NORM)', zorder=5)
for x, y, label in zip(x_aging, y_aging, labels_aging):
    ax.annotate(label, (x, y), fontsize=7, ha='center', va='top',
                xytext=(0, -12), textcoords='offset points')

for cls in ['STTC', 'MI', 'HYP', 'CD']:
    sub = df[df.clean_superclass == cls]
    ax.scatter(sub.age.mean(), sub.beta_mean.median(),
              s=200, c=COLORS[cls], marker='D', edgecolors='black', linewidth=0.5,
              zorder=6, label=f'{cls} (\u03b2={sub.beta_mean.median():.2f})')

ax.annotate('', xy=(80, 1.72), xytext=(30, 1.78),
            arrowprops=dict(arrowstyle='->', color=COLORS['NORM'], lw=2))
ax.text(55, 1.69, '\u03b2 \u2193 with age', color=COLORS['NORM'], fontsize=10, ha='center', fontweight='bold')
ax.annotate('', xy=(55, 2.05), xytext=(55, 1.85),
            arrowprops=dict(arrowstyle='->', color=COLORS['CD'], lw=2))
ax.text(58, 1.96, '\u03b2 \u2191 in disease', color=COLORS['CD'], fontsize=10, ha='left', fontweight='bold')

ax.set_xlabel('Mean age (years)')
ax.set_ylabel('\u03b2 (median)')
ax.set_title('C. Bifurcation: aging \u2193 vs. pathology \u2191', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

plt.suptitle('Figure 3. Aging and pathology drive \u03b2 in opposite directions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Figure 4: Beat-to-beat Variability (confirms aging story)
# ============================================================

# Load precomputed beat variability
beat_df = pd.read_csv(os.path.join(PRECOMPUTED_DIR, 'block_D_beat_variability.csv'))
print(f'Loaded beat variability for {len(beat_df):,} records')

bv = beat_df.dropna(subset=['II_morph_var', 'age']).copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel A: Morph variability vs age ---
ax = axes[0]
bins_bv = np.arange(20, 91, 5)
bv['age_bin'] = pd.cut(bv.age, bins=bins_bv)
binned_bv = bv.groupby('age_bin').agg(
    age_mid=('age', 'mean'), mv_mean=('II_morph_var', 'mean'),
    mv_se=('II_morph_var', lambda x: x.std() / np.sqrt(len(x))),
).dropna()

ax.scatter(bv.age, bv.II_morph_var, alpha=0.1, s=8, c='steelblue', rasterized=True)
ax.errorbar(binned_bv.age_mid, binned_bv.mv_mean, yerr=binned_bv.mv_se * 1.96,
            fmt='ko-', markersize=5, linewidth=1.5, capsize=3, zorder=5)
rho_mv, p_mv = stats.spearmanr(bv.age, bv.II_morph_var)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Morphological variability\n(1 \u2212 mean template correlation)')
ax.set_title(f'A. Beat variability rises with age\n\u03c1 = {rho_mv:.3f}, p = {p_mv:.1e}', fontweight='bold')
ax.grid(alpha=0.3)

# --- Panel B: Morph variability vs beta ---
ax = axes[1]
bv_beta = bv.dropna(subset=['beta_mean'])
ax.scatter(bv_beta.beta_mean, bv_beta.II_morph_var, alpha=0.1, s=8, c='coral', rasterized=True)
rho_bm, p_bm = stats.spearmanr(bv_beta.beta_mean, bv_beta.II_morph_var)
ax.set_xlabel('\u03b2 (spectral exponent)')
ax.set_ylabel('Morphological variability')
ax.set_title(f'B. Beat variability vs. \u03b2\n\u03c1 = {rho_bm:.3f}, p = {p_bm:.1e}', fontweight='bold')
ax.grid(alpha=0.3)

# --- Panel C: Schematic ---
ax = axes[2]
np.random.seed(42)
t = np.linspace(0, 0.6, 300)
template = np.exp(-(t - 0.2)**2 / 0.001) * 1.5 - np.exp(-(t - 0.22)**2 / 0.003) * 0.3 + \
           np.exp(-(t - 0.35)**2 / 0.005) * 0.3

for i in range(6):
    noise = np.random.randn(len(t)) * 0.03
    ax.plot(t, template + noise + i * 0.01, color=COLORS['NORM'], alpha=0.5, linewidth=0.8)
for i in range(6):
    noise = np.random.randn(len(t)) * 0.12
    shift = np.random.randn() * 0.02
    ax.plot(t + 0.7, template * (1 + np.random.randn() * 0.1) + noise + shift,
            color='#E53935', alpha=0.5, linewidth=0.8)

ax.axvline(0.65, color='gray', ls='--', alpha=0.5)
ax.text(0.3, ax.get_ylim()[1] * 0.9, 'Young heart\n(consistent)', ha='center', fontsize=10,
        color=COLORS['NORM'], fontweight='bold')
ax.text(1.0, ax.get_ylim()[1] * 0.9, 'Aging heart\n(variable)', ha='center', fontsize=10,
        color='#E53935', fontweight='bold')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title('C. Beat consistency declines with age', fontweight='bold')
ax.set_xticks([])

plt.suptitle('Figure 4. Beat-to-beat variability confirms complexity loss',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Key finding #3

- **β declines with healthy aging** (ρ = −0.18, p < 10⁻⁶⁰) — the aperiodic component flattens, reflecting loss of long-range temporal correlations
- **Pathology drives β up** — structural lesions create rigid, over-correlated dynamics that steepen the spectrum
- **Spatial heterogeneity (β_std) accelerates** after mid-life — marking the transition where age-related spectral degradation speeds up
- **Beat-to-beat morphological variability** independently confirms the story: individual heartbeats become less consistent with age (ρ = 0.165, p ≈ 10⁻¹⁹)
- Two metrics, two timescales (spectral = broadband, beats = per-cycle), same conclusion: **the aging heart loses electrical coordination**

---
## Result 4: Subclinical Detection — β Catches What the Expert Doesn't Diagnose

Among ECGs labeled "normal" by cardiologists, some have mild conduction findings noted but not diagnosed (IRBBB, IVCD, mild fascicular blocks). Can β distinguish these **subclinical** records from truly clean normals?

This tests β as an **early detection marker**: if spectral features shift *before* the cardiologist makes a diagnosis, β captures pre-clinical pathophysiology.

In [ ]:
# ============================================================
# Subclinical Detection: Pure NORM vs NORM + subclinical CD
# ============================================================

subclinical_cd_codes = {'IRBBB', 'IVCD', '1AVB', 'LAFB', 'LPFB', 'ILBBB'}

group_a_ids = []  # Pure NORM (only NORM code)
group_b_ids = []  # NORM + subclinical CD

for ecg_id, row in df.iterrows():
    codes = row['scp_dict']
    if 'NORM' not in codes or codes.get('NORM', 0) < 80:
        continue
    cd_present = {c for c in codes if c in subclinical_cd_codes and codes[c] > 0}
    if len(cd_present) == 0 and len(codes) == 1:
        group_a_ids.append(ecg_id)
    elif len(cd_present) > 0:
        group_b_ids.append(ecg_id)

a_data = df.loc[df.index.isin(group_a_ids)].copy()
b_data = df.loc[df.index.isin(group_b_ids)].copy()
a_data['sub_group'] = 0
b_data['sub_group'] = 1
combined = pd.concat([a_data, b_data])

print(f'Group A (Pure NORM): {len(a_data):,}')
print(f'Group B (NORM + subclinical CD): {len(b_data):,}')

# --- Statistical comparison ---
key_feats = ['beta_mean', 'beta_std', 'beta_anterior', 'beta_lateral',
             'beta_inferior', 'beta_regional_div', 'r2_mean']

effect_results = []
for feat in key_feats:
    va, vb = a_data[feat].dropna(), b_data[feat].dropna()
    _, p = stats.mannwhitneyu(va, vb)
    d = (vb.mean() - va.mean()) / np.sqrt(((len(va)-1)*va.std()**2 + (len(vb)-1)*vb.std()**2) / (len(va)+len(vb)-2))
    effect_results.append({'feature': feat, 'd': d, 'p': p, 'mean_A': va.mean(), 'mean_B': vb.mean()})
    star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  {feat:22s}: d = {d:+.3f}, p = {p:.1e} {star}')

# --- Classification ---
sub_feat_cols = get_feature_cols() if 'get_feature_cols' in dir() else feat_cols
sub_avail = [f for f in sub_feat_cols if f in combined.columns]

train_sub = combined[combined.strat_fold.isin(range(1, 9))]
test_sub = combined[combined.strat_fold.isin([9, 10])]

tr_s = train_sub[sub_avail + ['sub_group']].dropna()
te_s = test_sub[sub_avail + ['sub_group']].dropna()
X_tr_sub, y_tr_sub = tr_s[sub_avail].values, tr_s['sub_group'].values
X_te_sub, y_te_sub = te_s[sub_avail].values, te_s['sub_group'].values

sc_sub = StandardScaler()
X_tr_sub_s = sc_sub.fit_transform(X_tr_sub)
X_te_sub_s = sc_sub.transform(X_te_sub)

gbm_sub = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                      subsample=0.8, random_state=42)
gbm_sub.fit(X_tr_sub_s, y_tr_sub)
y_prob_sub = gbm_sub.predict_proba(X_te_sub_s)[:, 1]
sub_auc = roc_auc_score(y_te_sub, y_prob_sub)
sub_fpr, sub_tpr, _ = roc_curve(y_te_sub, y_prob_sub)
print(f'\nSubclinical detection AUC = {sub_auc:.3f} (test n={len(y_te_sub)})')

In [ ]:
# ============================================================
# Figure 5: Subclinical Detection
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel A: β distributions ---
ax = axes[0]
ax.hist(a_data.beta_mean.dropna(), bins=60, alpha=0.6, density=True,
        color=COLORS['NORM'], label=f'Pure NORM (n={len(a_data):,})', edgecolor='white', linewidth=0.3)
ax.hist(b_data.beta_mean.dropna(), bins=40, alpha=0.6, density=True,
        color='#E53935', label=f'NORM + subclinical CD (n={len(b_data):,})', edgecolor='white', linewidth=0.3)
ax.axvline(a_data.beta_mean.median(), color=COLORS['NORM'], ls='--', lw=2,
           label=f'Median = {a_data.beta_mean.median():.3f}')
ax.axvline(b_data.beta_mean.median(), color='#E53935', ls='--', lw=2,
           label=f'Median = {b_data.beta_mean.median():.3f}')
ax.set_xlabel('\u03b2 (mean across leads)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('A. \u03b2 distribution: pure NORM vs subclinical CD', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Panel B: Effect sizes ---
ax = axes[1]
eff_df = pd.DataFrame(effect_results).sort_values('d', ascending=True)
colors_eff = ['#E53935' if abs(d) > 0.3 else '#FF9800' if abs(d) > 0.15 else '#9E9E9E'
              for d in eff_df.d.values]
ax.barh(range(len(eff_df)), eff_df.d.values, color=colors_eff, alpha=0.85)
ax.set_yticks(range(len(eff_df)))
ax.set_yticklabels(eff_df.feature.values, fontsize=9)
ax.axvline(0, color='black', lw=0.5)
ax.axvline(-0.2, color='gray', ls=':', alpha=0.5, label='small (0.2)')
ax.axvline(0.2, color='gray', ls=':', alpha=0.5)
ax.set_xlabel("Cohen's d (subclinical \u2212 pure NORM)", fontsize=10)
ax.set_title('B. Effect sizes: subclinical shifts \u03b2 down', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# Stars
for i, (_, row) in enumerate(eff_df.iterrows()):
    star = '***' if row['p'] < 0.001 else '**' if row['p'] < 0.01 else '*' if row['p'] < 0.05 else ''
    if star:
        x_pos = row['d'] + (0.01 if row['d'] >= 0 else -0.06)
        ax.text(x_pos, i, star, va='center', fontsize=10, fontweight='bold', color='#C62828')

# --- Panel C: ROC ---
ax = axes[2]
ax.plot(sub_fpr, sub_tpr, linewidth=2.5, color='#C62828',
        label=f'GBM (AUC = {sub_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'C. Subclinical detection: AUC = {sub_auc:.3f}\n'
             f'Can \u03b2 catch what the cardiologist doesn\'t diagnose?', fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)

plt.suptitle('Figure 5. Subclinical conduction disturbance detection',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Key finding #4

- **β distinguishes pure normals from normals with subclinical conduction findings** (AUC ≈ 0.76)
- The strongest effect is in **beta_anterior** (d = −0.55, medium effect size) — exactly where conduction disturbances manifest
- **Direction is inverted relative to overt CD:** subclinical CD → β *decreases* (fragmentation), while overt CD → β *increases* (rigid slowing)
- This makes physical sense: incomplete block = electrical fragmentation without a dominant slow pathway (noisy, β↓), while complete block = one big slow zone (smooth, β↑)
- **β captures a gradient of conduction disturbance severity**, from subclinical fragmentation through normal to overt rigid blockade
- These patients were labeled "normal" by the reading cardiologist — β detects the subclinical signal they chose not to diagnose

---
## Summary Statistics

In [ ]:
# ============================================================
# Summary table
# ============================================================
print('=' * 70)
print('SUMMARY OF FINDINGS')
print('=' * 70)

print(f'\nDataset: PTB-XL, {len(df):,} records, 12-lead ECG @ 500 Hz')
print(f'NORM cohort: {len(norm):,} healthy records (18-95 years)')
print(f'\u03b2 reference: NORM median = {norm.beta_mean.median():.3f}')

print(f'\n--- Result 1: Diagnostic Landscape ---')
clean = df[df.clean_superclass.isin(['NORM','MI','STTC','CD','HYP'])].copy()
norm_vals = clean[clean.clean_superclass == 'NORM'].beta_mean.dropna()
n1, s1 = len(norm_vals), norm_vals.std()
for cls in ['MI', 'STTC', 'HYP', 'CD']:
    cls_vals = clean[clean.clean_superclass == cls].beta_mean.dropna()
    n2, s2 = len(cls_vals), cls_vals.std()
    pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
    d = (cls_vals.mean() - norm_vals.mean()) / pooled
    print(f"  {cls:6s}: Cohen's d = {d:.3f}")

print(f'\n--- Result 2: Spectral Anatomy ---')
print(f'  CLBBB vs CRBBB: AUC = {bbb_auc:.3f}')
print(f'  6-class CD subtype: Macro AUC = {macro_auc6:.3f}')
for cls in cd_subtypes_of_interest:
    if cls in per_class_auc6:
        print(f'    {cls:8s}: AUC = {per_class_auc6[cls]:.3f}')

print(f'\n--- Result 3: Aging Bifurcation ---')
print(f'  \u03b2 vs age (NORM): \u03c1 = {rho_age:.4f}')
print(f'  Spatial heterogeneity breakpoint: ~{bp:.0f} years')
print(f'  Beat variability vs age: \u03c1 = {rho_mv:.4f}')

print(f'\n--- Result 4: Subclinical Detection ---')
print(f'  Pure NORM vs NORM+subclinical CD: AUC = {sub_auc:.3f}')
for r in sorted(effect_results, key=lambda x: abs(x['d']), reverse=True)[:3]:
    print(f"  {r['feature']:22s}: d = {r['d']:+.3f}, p = {r['p']:.1e}")

print('\n' + '=' * 70)

---
## Discussion

### What β tells us about the heart

The spectral exponent β is a single number that captures the balance between order and disorder in cardiac electrical activity. The healthy heart operates at β ≈ 1.76 — not at theoretical criticality (β = 1.0), but at a stable operating point from which deviations signal trouble.

### The three regimes

Our results reveal three distinct β-regimes:

1. **Subclinical fragmentation (β ↓↓):** Incomplete conduction disturbances fragment the electrical wavefront without creating a dominant slow pathway. The spectrum flattens toward noise.
2. **Normal range (β ≈ 1.76):** The healthy operating point. Coordinated propagation with optimal balance of deterministic and stochastic dynamics.
3. **Rigid blockade (β ↑↑):** Complete bundle branch blocks create deterministic re-entry patterns that steepen the spectrum. The blocked region forces over-correlated dynamics.

Aging naturally pushes toward regime 1 (β↓), while overt pathology pushes toward regime 3 (β↑). This bifurcation is the central organizing principle.

### Why the spectral anatomy result matters

Distinguishing CLBBB from CRBBB at AUC = 0.984 using only spectral exponents — without any waveform morphology features — demonstrates that β encodes genuine electrophysiology, not statistical artifacts. The fact that feature importance maps directly onto cardiac anatomy (V1-V4 for left bundle, V1 for right bundle) confirms that we're measuring the physics of conduction, not fitting noise.

This doesn't compete with deep learning for clinical deployment. Instead, it reveals *what* deep learning implicitly extracts: the aperiodic structure of the power spectrum carries most of the diagnostic information about conduction disorders.

### Limitations

1. **10-second windows:** Short ECGs provide limited spectral resolution. Longer recordings would improve β estimates.
2. **Cross-sectional design:** We observe aging effects across individuals. Longitudinal follow-up would establish predictive validity for subclinical detection.
3. **Medication confounds:** PTB-XL lacks medication data. Beta-blockers and antiarrhythmics could affect β.
4. **Subclinical sample size:** The NORM+subclinical group (n ≈ 330) is relatively small. External validation would strengthen the subclinical detection claim.

### What's next

- **Longitudinal validation:** Do patients with subclinical β-deviation progress to overt conduction disorders?
- **Continuous monitoring:** Apply β to wearable/Holter data for real-time complexity tracking
- **Multi-scale integration:** Link ECG-scale β changes to molecular-level ion channel expression (gap junction genes Gja1/Gja5, sodium channel Scn5a)
- **DL complementarity:** Test whether β-features improve deep learning AUC when added as auxiliary inputs

---

*If you found this analysis useful, please upvote! All code is reproducible from the precomputed features.*

**References:**
- Wen H, Liu Z (2016). Separating fractal and oscillatory components in the power spectrum. *Brain Topography*, 29(1), 13–26.
- Wagner P et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.
- Strodthoff N et al. (2021). Deep learning for ECG analysis: Benchmarks and insights from PTB-XL. *IEEE JBHI*, 25(5), 1519–1528.
- Goldberger AL et al. (2002). Fractal dynamics in physiology: Alterations with disease and aging. *PNAS*, 99(suppl 1), 2466–2472.